# LUMINA — Pipeline AI: Preprocessing → Feature Engineering → Spatial XGBoost
## Adaptasi dengan Dataset Train.csv

**Tim Radiant · MAPID WebGIS Competition 2026**

Notebook ini mengadaptasi kerangka kerja LUMINA end-to-end untuk menghasilkan **indeks kepadatan (0–100) per sel H3 per slot waktu** menggunakan dataset `Train.csv` (data kepadatan stasiun/kereta sebagai *staging/dummy dataset*).

**Tahapan yang dicakup:**
1. Data Ingestion (`Train.csv`)
2. Cleaning & Preprocessing
3. Validasi & Audit Data
4. Spatial Join → Grid H3 Resolusi 9
5. Feature Engineering Dua Dimensi (sel × slot waktu) + Variabel Ketetanggaan Spasial
6. Normalisasi & Indeks Proksi Komposit
7. Penggabungan Label → Dataset Modeling
8. Split 80:20 + Training **Spatial XGBoost** (+ Dua Baseline Pembanding)
9. Block Spatial Cross-Validation
10. SHAP Explainability (Faktor Pendorong Panel AI)
11. Indeks Komposit: Skor Potensi Lokasi & Indeks Risiko
12. Aturan Rekomendasi Kategori Usaha (Fitur 7)
13. Packaging Output → JSON (siap dikonsumsi REST API / AI Explanation Layer)
14. Catatan: Kenapa PCA tidak dipakai di pipeline ini
15. Ringkasan & Pemetaan ke Timeline PRD (M1–M8)

## 0. Setup & Import

In [1]:
import h3
import xgboost as xgb
import xgboost
import shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, RidgeCV, Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split, GroupKFold
from xgboost import XGBRegressor

try:
    display
except NameError:
    display = print

rng = np.random.default_rng(42)
pd.set_option("display.width", 140)
print("Setup selesai. h3:", h3.__version__ if hasattr(h3, "__version__") else "ok",
      "| xgboost:", xgboost.__version__, "| shap:", shap.__version__)


Setup selesai. h3: 4.5.0 | xgboost: 3.4.1 | shap: 0.52.0


## 1. Data Ingestion — Train.csv

Dataset `Train.csv` berisi data perjalanan kereta dengan kolom `target` (high/medium/low) yang merepresentasikan tingkat kepadatan. Dataset ini digunakan sebagai *staging dataset* untuk menguji logika pipeline (skema, cleaning, spatial join H3, feature engineering, modeling) sebelum akses API MAPID resmi didapat.

> **Ganti di sini nanti:** begitu API MAPID & data survei siap, ubah `pd.read_csv('Train.csv')` menjadi pemanggilan API asli — selama nama kolom output di-mapping dengan konsisten, semua tahap berikutnya tetap berjalan tanpa perubahan.

In [2]:
raw_df = pd.read_csv('data/staging/Train.csv')
print("Ukuran dataset raw:", raw_df.shape)
print("Daftar kolom:", list(raw_df.columns))
print()
print("Distribusi target:")
print(raw_df['target'].value_counts())
print()
raw_df.head()

Ukuran dataset raw: (1284, 19)
Daftar kolom: ['id_code', 'current_date', 'current_time', 'source_name', 'destination_name', 'train_name', 'target', 'country_code_source', 'longitude_source', 'latitude_source', 'mean_halt_times_source', 'country_code_destination', 'longitude_destination', 'latitude_destination', 'mean_halt_times_destination', 'current_year', 'current_week', 'current_day', 'is_weekend']

Distribusi target:
target
low       532
high      392
medium    360
Name: count, dtype: int64

Out[0]: 
           id_code current_date  ... current_day is_weekend
0  isfywypmkqqhyft   2016-07-27  ...   Wednesday      False
1  mqsfxyvuqpbwomk   2016-07-27  ...   Wednesday      False
2  alspwwtbdvqsgby   2016-07-27  ...   Wednesday      False
3  szitxhhqduyrqpg   2016-07-27  ...   Wednesday      False
4  krisdqzczivvwcp   2016-07-27  ...   Wednesday      False

[5 rows x 19 columns]


## 2. Cleaning & Preprocessing

Langkah cleaning & penyiapan data:
1. Menghapus baris dengan koordinat stasiun asal (`latitude_source`, `longitude_source`) yang kosong.
2. Mengisi nilai NaN pada kolom destinasi dengan imputasi yang sesuai (mean untuk variabel numerik, mode untuk kategorikal).
3. Melakukan mapping nama kolom ke standar LUMINA (`latitude_source` → `lat`, `longitude_source` → `lon`, `source_name` → `station_ref`).
4. Memproses timestamp (`current_date` + `current_time`) dan mengelompokkan ke dalam slot waktu (pagi sibuk, siang, sore sibuk, malam).
5. Mengubah label kategorikal `target` ('low', 'medium', 'high') ke skala kontinu 0–100 (`density_label`).

In [3]:
df = raw_df.dropna(subset=['longitude_source', 'latitude_source', 'country_code_source']).copy()

for col in ['longitude_destination', 'latitude_destination', 'mean_halt_times_destination']:
    df[col] = df[col].fillna(df[col].mean())
df['country_code_destination'] = df['country_code_destination'].fillna(df['country_code_destination'].mode()[0])

df = df.rename(columns={
    'source_name': 'station_ref',
    'latitude_source': 'lat',
    'longitude_source': 'lon',
})

def parse_time(row):
    from datetime import datetime
    dt_str = f"{row['current_date']} {row['current_time']}"
    try:
        return datetime.strptime(dt_str, "%Y-%m-%d %I:%M:%S %p")
    except ValueError:
        return pd.NaT

df['timestamp'] = df.apply(parse_time, axis=1)
df = df.dropna(subset=['timestamp'])

def to_time_slot(ts):
    hour = ts.hour
    if 6 <= hour <= 9:   return "pagi_sibuk_06_09"
    if 10 <= hour <= 15:  return "siang_10_15"
    if 16 <= hour <= 19:  return "sore_sibuk_16_19"
    return "malam"

df['time_slot'] = df['timestamp'].apply(to_time_slot)

TARGET_MAP = {"low": 0.0, "medium": 50.0, "high": 100.0}
df['density_label'] = df['target'].map(TARGET_MAP)

print(f"Setelah cleaning & preprocessing: {df.shape}")
print("Distribusi time_slot:")
print(df['time_slot'].value_counts())
print()
df[['station_ref', 'lat', 'lon', 'time_slot', 'target', 'density_label']].head()


Setelah cleaning & preprocessing: (1283, 22)
Distribusi time_slot:
time_slot
sore_sibuk_16_19    404
pagi_sibuk_06_09    355
siang_10_15         313
malam               211
Name: count, dtype: int64

Out[0]: 
   station_ref        lat       lon time_slot target  density_label
0  station$147  50.845658  4.356801     malam   high          100.0
1  station$147  50.845658  4.356801     malam   high          100.0
2  station$147  50.845658  4.356801     malam   high          100.0
3  station$147  50.845658  4.356801     malam   high          100.0
4  station$147  50.845658  4.356801     malam   high          100.0


## 3. Validasi & Audit Data

Memastikan integritas dataset:
- Bebas dari missing values pada kolom koordinat, timestamp, dan label target.
- Pengecekan kewajaran rentang koordinat dan distribusi variabel numerik (`mean_halt_times_source`).

In [4]:
assert df[['lat', 'lon', 'timestamp', 'density_label']].isnull().sum().sum() == 0, "Ada data kunci kosong!"

print("Statistik deskriptif mean_halt_times_source:")
print(df['mean_halt_times_source'].describe())
print()
print("Rentang koordinat:")
print(f"  Latitude : {df['lat'].min():.4f} s/d {df['lat'].max():.4f}")
print(f"  Longitude: {df['lon'].min():.4f} s/d {df['lon'].max():.4f}")
print()
print("Jumlah stasiun unik:", df['station_ref'].nunique())
print("Jumlah nama kereta unik:", df['train_name'].nunique())

Statistik deskriptif mean_halt_times_source:
count    1283.000000
mean      278.061613
std       228.954089
min         0.000000
25%        78.488439
50%       180.598266
75%       467.982659
max       686.615607
Name: mean_halt_times_source, dtype: float64

Rentang koordinat:
  Latitude : 49.6385 s/d 51.9251
  Longitude: -0.1261 s/d 5.9823

Jumlah stasiun unik: 187
Jumlah nama kereta unik: 559


## 4. Spatial Join → Grid H3 Resolusi 9

Setiap titik koordinat (lat, lon) dipetakan ke sel H3 resolusi 9 (± 0,10 km²) menggunakan `h3.latlng_to_cell`. Grid H3 ini berfungsi sebagai unit analisis spasial tunggal yang menyatukan seluruh fitur dan sinyal kepadatan.

In [5]:
H3_RES = 9

df['h3_cell'] = df.apply(lambda r: h3.latlng_to_cell(r['lat'], r['lon'], H3_RES), axis=1)

print("Jumlah sel H3 unik yang tersentuh data:", df['h3_cell'].nunique())
df[['station_ref', 'lat', 'lon', 'h3_cell', 'time_slot', 'density_label']].head()

Jumlah sel H3 unik yang tersentuh data: 187
Out[0]: 
   station_ref        lat       lon          h3_cell time_slot  density_label
0  station$147  50.845658  4.356801  891fa44180fffff     malam          100.0
1  station$147  50.845658  4.356801  891fa44180fffff     malam          100.0
2  station$147  50.845658  4.356801  891fa44180fffff     malam          100.0
3  station$147  50.845658  4.356801  891fa44180fffff     malam          100.0
4  station$147  50.845658  4.356801  891fa44180fffff     malam          100.0


## 5. Feature Engineering — Dua Dimensi (sel × slot waktu)

Sesuai PRD LUMINA, fitur dibagi menjadi dua kelompok:
1. **Fitur Temporal**: Berosilasi per `(h3_cell, time_slot)` (intensitas aktivitas, durasi transit/halt, keberagaman rute temporal).
2. **Fitur Kawasan**: Bersifat statis per `h3_cell` (kepadatan total perjalanan, keberagaman rute, keberagaman armada kereta, rerata durasi transit).

Ditambah **variabel ketetanggaan spasial**: nilai rerata fitur pada sel-sel tetangga langsung (`h3.grid_disk(cell, k=1)`).

In [6]:
temporal = df.groupby(['h3_cell', 'time_slot']).agg(
    activity_intensity_raw=('id_code', 'count'),
    halt_time_signal_raw=('mean_halt_times_source', 'mean'),
    route_diversity_temporal_raw=('destination_name', 'nunique'),
).reset_index()

def minmax(s):
    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)
    return (s - s.min()) / (s.max() - s.min()) * 100

for col in ['activity_intensity', 'halt_time_signal', 'route_diversity_temporal']:
    temporal[col] = minmax(temporal[f'{col}_raw'])

print("Tabel fitur TEMPORAL (contoh):")
display(temporal[['h3_cell', 'time_slot', 'activity_intensity', 'halt_time_signal', 'route_diversity_temporal']].head())

kawasan = df.groupby('h3_cell').agg(
    trip_density_raw=('id_code', 'count'),
    route_diversity_raw=('destination_name', 'nunique'),
    train_diversity_raw=('train_name', 'nunique'),
    mean_halt_time_raw=('mean_halt_times_source', 'mean'),
).reset_index()

for col in ['trip_density', 'route_diversity', 'train_diversity', 'mean_halt_time']:
    kawasan[col] = minmax(kawasan[f'{col}_raw'])

print("\nTabel fitur KAWASAN (contoh):")
display(kawasan[['h3_cell', 'trip_density', 'route_diversity', 'train_diversity', 'mean_halt_time']].head())

Tabel fitur TEMPORAL (contoh):
           h3_cell  ... route_diversity_temporal
0  89194d01c3bffff  ...                 0.000000
1  89194d01c3bffff  ...                 4.166667
2  89194d04bc7ffff  ...                 0.000000
3  89194d05cc3ffff  ...                 0.000000
4  89194d05cc3ffff  ...                 0.000000

[5 rows x 5 columns]

Tabel fitur KAWASAN (contoh):
           h3_cell  trip_density  ...  train_diversity  mean_halt_time
0  89194d01c3bffff      1.739130  ...         2.531646        7.732911
1  89194d04bc7ffff      0.869565  ...         0.000000        3.866666
2  89194d05cc3ffff      0.869565  ...         1.265823        4.835648
3  89194d0e06fffff      4.347826  ...         6.329114        5.137455
4  89194d115bbffff     32.173913  ...        34.177215       23.946306

[5 rows x 5 columns]


In [7]:
def neighbor_features(df_in, value_cols, key_cols, rings=[1, 2]):
    """Menghitung rata-rata nilai ketetanggaan spasial pada ring heksagon H3 berorde k=1 dan k=2."""
    df_out = df_in.copy()
    lookup = df_in.set_index(key_cols)[value_cols]
    
    for k in rings:
        results = []
        for _, row in df_in.iterrows():
            disk = [c for c in h3.grid_disk(row['h3_cell'], k) if c != row['h3_cell']]
            if len(key_cols) == 2:
                sub = lookup.reindex([(c, row[key_cols[1]]) for c in disk])
            else:
                sub = lookup.reindex(disk)
            results.append(sub.mean().fillna(0).values)
        arr = np.array(results)
        for i, c in enumerate(value_cols):
            suffix = '_nbr' if k == 1 else f'_nbr_k{k}'
            df_out[f'{c}{suffix}'] = arr[:, i]
    return df_out

TEMPORAL_FEATURES = ['activity_intensity', 'halt_time_signal', 'route_diversity_temporal']
KAWASAN_FEATURES = ['trip_density', 'route_diversity', 'train_diversity', 'mean_halt_time']

temporal = neighbor_features(temporal, TEMPORAL_FEATURES, ['h3_cell', 'time_slot'], rings=[1, 2])
kawasan = neighbor_features(kawasan, KAWASAN_FEATURES, ['h3_cell'], rings=[1, 2])

print("Fitur spasial multiskala (Ring 1 & Ring 2) berhasil ditambahkan:")
display(temporal[['h3_cell', 'time_slot', 'activity_intensity', 'activity_intensity_nbr', 'activity_intensity_nbr_k2']].head())


Fitur spasial multiskala (Ring 1 & Ring 2) berhasil ditambahkan:
           h3_cell  ... activity_intensity_nbr_k2
0  89194d01c3bffff  ...                       0.0
1  89194d01c3bffff  ...                       0.0
2  89194d04bc7ffff  ...                       0.0
3  89194d05cc3ffff  ...                       0.0
4  89194d05cc3ffff  ...                       0.0

[5 rows x 5 columns]


## 6. Normalisasi & Indeks Proksi Komposit

Tiga sinyal proksi temporal digabung menjadi satu indeks 0–100 per `(h3_cell, time_slot)`. Bobot sama rata (1/3 masing-masing) digunakan sebagai *placeholder* yang dapat dikalibrasi sesuai domain pengetahuan bisnis.

In [8]:
W_PROXY = {"activity_intensity": 1/3, "halt_time_signal": 1/3, "route_diversity_temporal": 1/3}
temporal['density_index_proxy'] = sum(temporal[k] * w for k, w in W_PROXY.items())
display(temporal[['h3_cell', 'time_slot', 'density_index_proxy']].head())

           h3_cell         time_slot  density_index_proxy
0  89194d01c3bffff  pagi_sibuk_06_09             2.577637
1  89194d01c3bffff  sore_sibuk_16_19             4.595457
2  89194d04bc7ffff  pagi_sibuk_06_09             1.917820
3  89194d05cc3ffff  pagi_sibuk_06_09             1.611883
4  89194d05cc3ffff  sore_sibuk_16_19             1.611883


## 7. Menggabungkan Label → Dataset Modeling

Nilai observasi `density_label` (skala 0–100) dirata-ratakan per `(h3_cell, time_slot)` lalu dipasangkan dengan kumpulan fitur temporal dan kawasan. Matriks $(X, y)$ inilah yang digunakan untuk melatih Spatial XGBoost.

In [9]:
label_df = df.groupby(['h3_cell', 'time_slot'])['density_label'].mean().reset_index()

model_df = label_df.merge(temporal, on=['h3_cell', 'time_slot'], how='left')
model_df = model_df.merge(
    kawasan.drop(columns=[c for c in kawasan.columns if c.endswith('_raw')]),
    on='h3_cell', how='left'
)
model_df = model_df.fillna(0)

# Feature Engineering Tingkat Lanjut: Rasio Spasial, Interaksi Non-Linear & Proksi Dinamika Kawasan
model_df["spatial_activity_ratio"] = model_df["activity_intensity"] / (model_df["activity_intensity_nbr"] + 1.0)
model_df["spatial_density_ratio"] = model_df["trip_density"] / (model_df["trip_density_nbr"] + 1.0)
model_df["trip_x_activity"] = (model_df["trip_density"] * model_df["activity_intensity"]) / 100.0
model_df["route_to_train_diversity"] = (model_df["route_diversity"] + 1.0) / (model_df["train_diversity"] + 1.0)
model_df["halt_intensity_synergy"] = (model_df["mean_halt_time"] * model_df["halt_time_signal"]) / 100.0
model_df["temporal_density_decay"] = model_df["trip_density"] * np.exp(-0.02 * model_df["mean_halt_time"])
model_df["network_centrality_proxy"] = 0.4 * model_df["trip_density"] + 0.3 * model_df["route_diversity"] + 0.3 * model_df["train_diversity"]

# H3 Parent Resolusi 7 sebagai batas spasial bebas leakage (Block ID)
model_df['block_id'] = model_df['h3_cell'].apply(lambda c: h3.cell_to_parent(c, H3_RES - 2))

FEATURES_BASE = (
    TEMPORAL_FEATURES + [f'{c}_nbr' for c in TEMPORAL_FEATURES] +
    KAWASAN_FEATURES + [f'{c}_nbr' for c in KAWASAN_FEATURES]
)
FEATURES_ADVANCED = FEATURES_BASE + [
    f'{c}_nbr_k2' for c in TEMPORAL_FEATURES
] + [
    f'{c}_nbr_k2' for c in KAWASAN_FEATURES
] + [
    "density_index_proxy",
    "spatial_activity_ratio", "spatial_density_ratio", "trip_x_activity",
    "route_to_train_diversity", "halt_intensity_synergy", "temporal_density_decay",
    "network_centrality_proxy"
]

X = model_df[FEATURES_ADVANCED]
X_base = model_df[FEATURES_BASE]
X_no_nbr = model_df[[c for c in FEATURES_BASE if not c.endswith('_nbr')]]
y = model_df['density_label']
groups = model_df['block_id']

print("Ukuran dataset modeling (baris x kolom):", model_df.shape)
print("Jumlah fitur dasar:", len(FEATURES_BASE))
print("Jumlah fitur setelah Advanced Feature Engineering:", len(FEATURES_ADVANCED))
display(model_df[['h3_cell', 'time_slot', 'density_label', 'spatial_activity_ratio', 'trip_x_activity', 'network_centrality_proxy']].head())


Ukuran dataset modeling (baris x kolom): (356, 36)
Jumlah fitur dasar: 14
Jumlah fitur setelah Advanced Feature Engineering: 29
           h3_cell         time_slot  ...  trip_x_activity  network_centrality_proxy
0  89194d01c3bffff  pagi_sibuk_06_09  ...         0.000000                  2.422888
1  89194d01c3bffff  sore_sibuk_16_19  ...         0.032814                  2.422888
2  89194d04bc7ffff  pagi_sibuk_06_09  ...         0.016407                  0.347826
3  89194d05cc3ffff  pagi_sibuk_06_09  ...         0.000000                  1.695315
4  89194d05cc3ffff  sore_sibuk_16_19  ...         0.000000                  1.695315

[5 rows x 6 columns]


## 8. Split 80:20 + Model Utama: Spatial XGBoost (+ Dua Baseline Pembanding)

Model utama yang digunakan adalah **XGBoost Regressor** (dengan fitur ketetanggaan spasial) untuk memprediksi indeks kepadatan kontinu (0–100).
Dua model baseline pembanding:
1. **Model Naif**: Rata-rata historis
2. **Regresi Linier**: Tanpa fitur ketetanggaan spasial

In [10]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
Xb_train, Xb_test = X_base.loc[X_train.index], X_base.loc[X_test.index]
Xn_train, Xn_test = X_no_nbr.loc[X_train.index], X_no_nbr.loc[X_test.index]

# 1. Baseline Naif
naive_pred = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

# 2. Linear Regression (tanpa neighbor)
lr = LinearRegression().fit(Xn_train, y_train)
pred_lr = lr.predict(Xn_test)

# 3. Spatial XGBoost (Base Features)
xgb_base = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.08,
                        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, random_state=42)
xgb_base.fit(Xb_train, y_train)
pred_xgb_base = xgb_base.predict(Xb_test)

# 4. Spatial XGBoost (+ Advanced Feature Engineering)
xgb_feat = XGBRegressor(n_estimators=150, max_depth=3, learning_rate=0.08,
                        subsample=0.8, colsample_bytree=0.8, reg_lambda=2.0, random_state=42)
xgb_feat.fit(X_train, y_train)
pred_xgb_feat = xgb_feat.predict(X_test)

def calc_metrics(name, y_true, y_pred):
    return {
        "Model": name,
        "MAE": round(mean_absolute_error(y_true, y_pred), 2),
        "RMSE": round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        "R²": round(r2_score(y_true, y_pred), 3)
    }

eval_df = pd.DataFrame([
    calc_metrics("1. Naive Baseline (Rata-rata)", y_test, naive_pred),
    calc_metrics("2. Linear Reg (Tanpa Spatial Lag)", y_test, pred_lr),
    calc_metrics("3. Spatial XGBoost (Base Features)", y_test, pred_xgb_base),
    calc_metrics("4. Spatial XGBoost (+Feature Engineering)", y_test, pred_xgb_feat)
])

print("== Evaluasi Holdout 80:20 Awal (Pengaruh Feature Engineering) ==")
display(eval_df)


== Evaluasi Holdout 80:20 Awal (Pengaruh Feature Engineering) ==
                                       Model    MAE   RMSE     R²
0              1. Naive Baseline (Rata-rata)  29.26  34.95 -0.011
1          2. Linear Reg (Tanpa Spatial Lag)  29.23  34.68  0.005
2         3. Spatial XGBoost (Base Features)  27.94  33.14  0.091
3  4. Spatial XGBoost (+Feature Engineering)  27.76  32.60  0.120


## 9. Block Spatial Cross-Validation, Fine-Tuning Hyperparameter & Pencegahan Over/Under-Fitting

### 🧠 Perspektif AI & Machine Learning Engineer:
> **Apakah Spatial K-Fold dapat meningkatkan akurasi, memberikan insight mendalam, dan mencegah over/under-fitting?**
>
> 1. **Mekanisme Peningkatan Akurasi:**
>    - Secara teoritis, K-Fold adalah protokol *estimasi generalisasi*, bukan optimizer bobot gradien. Namun, K-Fold menghasilkan prediksi **Out-Of-Fold (OOF)** yang steril (bebas leakage). Prediksi OOF inilah yang menjadi input fundamental untuk melatih **Stacking/Blending Meta-Learner (Ensemble)**, yang secara empiris terbukti menurunkan varians dan meningkatkan akurasi.
> 2. **Pencegahan Overfitting & Underfitting pada Domain Spasial:**
>    - **Tobler's First Law of Geography**: Objek yang dekat secara spasial memiliki keterkaitan lebih tinggi dibanding objek yang jauh (*Spatial Autocorrelation*). Random K-Fold biasa akan membocorkan fitur tetangga (`_nbr`) ke validation set (*data leakage* fatal), menghasilkan skor evaluasi yang optimis semu (ilusi akurasi).
>    - Dengan **Spatial Block K-Fold (Uber H3 Resolusi 7)**, fold dipisahkan berdasarkan klaster geografis makro. Train set dan Validation set tidak saling bertetangga.
>    - **Diagnosis Bias-Variance (Generalization Gap = Train R² - Val R²):**
>      - *Underfitting*: Train $R^2$ rendah, Val $R^2$ rendah (model gagal menangkap pola). Solusi: Tambahkan fitur interaksi & perluas kapasitas model.
>      - *Overfitting*: Train $R^2$ tinggi (>0.85), Val $R^2$ anjlok (Gap > 0.40). Solusi: Tambahkan regularisasi $L_1$ (`reg_alpha`) & $L_2$ (`reg_lambda`), perkecil `learning_rate`, dan batasi `max_depth` $\le 3$.
>      - *Optimal Fit*: Validation MAE minimal dengan Generalization Gap terkendali.


In [11]:
gkf = GroupKFold(n_splits=5)

grid_configs = [
    {"name": "Trial 1 (Conservative)", "n_estimators": 80, "max_depth": 2, "learning_rate": 0.05, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.1, "reg_lambda": 3.0},
    {"name": "Trial 2 (Moderate Shallow)", "n_estimators": 100, "max_depth": 2, "learning_rate": 0.06, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.05, "reg_lambda": 2.5},
    {"name": "Trial 3 (Balanced Depth 3)", "n_estimators": 120, "max_depth": 3, "learning_rate": 0.04, "subsample": 0.80, "colsample_bytree": 0.80, "reg_alpha": 0.1, "reg_lambda": 3.0},
    {"name": "Trial 4 (Low LR Regularized)", "n_estimators": 150, "max_depth": 2, "learning_rate": 0.04, "subsample": 0.85, "colsample_bytree": 0.80, "reg_alpha": 0.1, "reg_lambda": 3.0},
    {"name": "Trial 5 (Robust Regularized)", "n_estimators": 180, "max_depth": 3, "learning_rate": 0.03, "subsample": 0.85, "colsample_bytree": 0.85, "reg_alpha": 0.2, "reg_lambda": 4.0},
]

tune_records = []
oof_dict = {}

for cfg in grid_configs:
    params = {k: v for k, v in cfg.items() if k != "name"}
    tr_r2s, va_r2s, va_maes = [], [], []
    oof = np.zeros(len(X))
    
    for tr_i, val_i in gkf.split(X, y, groups=groups):
        X_tr, y_tr = X.iloc[tr_i], y.iloc[tr_i]
        X_va, y_va = X.iloc[val_i], y.iloc[val_i]
        
        m = XGBRegressor(**params, random_state=42)
        m.fit(X_tr, y_tr)
        
        p_tr = m.predict(X_tr)
        p_va = m.predict(X_va)
        oof[val_i] = p_va
        
        tr_r2s.append(r2_score(y_tr, p_tr))
        va_r2s.append(r2_score(y_va, p_va))
        va_maes.append(mean_absolute_error(y_va, p_va))
        
    mean_tr_r2 = np.mean(tr_r2s)
    mean_va_r2 = np.mean(va_r2s)
    mean_va_mae = np.mean(va_maes)
    gap = mean_tr_r2 - mean_va_r2
    
    tune_records.append({
        "Konfigurasi": cfg["name"],
        "Trees": params["n_estimators"],
        "Depth": params["max_depth"],
        "LR": params["learning_rate"],
        "Reg L1/L2": f"{params['reg_alpha']}/{params['reg_lambda']}",
        "Val MAE": round(mean_va_mae, 2),
        "Val R²": round(mean_va_r2, 3),
        "Train R²": round(mean_tr_r2, 3),
        "Generalization Gap": round(gap, 3),
        "Status Fitting": "Optimal (No Overfitting)" if gap < 0.40 else "Mild Overfitting"
    })
    oof_dict[cfg["name"]] = oof

tune_df = pd.DataFrame(tune_records).sort_values("Val MAE")
print("=== Hasil Fine-Tuning Hyperparameter via Spatial Block K-Fold ===")
display(tune_df)

best_cfg = grid_configs[4] # Trial 5 terpilih
best_params = {k: v for k, v in best_cfg.items() if k != "name"}
xgb_tuned = XGBRegressor(**best_params, random_state=42)
xgb_tuned.fit(X_train, y_train)
pred_xgb_tuned = xgb_tuned.predict(X_test)
print(f"\n[OK] Parameter Terbaik Terpilih: {best_cfg['name']} | Holdout Test MAE: {mean_absolute_error(y_test, pred_xgb_tuned):.2f}")


=== Hasil Fine-Tuning Hyperparameter via Spatial Block K-Fold ===
                    Konfigurasi  ...            Status Fitting
4  Trial 5 (Robust Regularized)  ...  Optimal (No Overfitting)
2    Trial 3 (Balanced Depth 3)  ...  Optimal (No Overfitting)
3  Trial 4 (Low LR Regularized)  ...  Optimal (No Overfitting)
0        Trial 1 (Conservative)  ...  Optimal (No Overfitting)
1    Trial 2 (Moderate Shallow)  ...  Optimal (No Overfitting)

[5 rows x 10 columns]

[OK] Parameter Terbaik Terpilih: Trial 5 (Robust Regularized) | Holdout Test MAE: 27.67


## 9.1 Multi-Model Spatial Ensemble (Blending & Stacking Meta-Learner)

Untuk melampaui limitasi representasi algoritma tunggal (*single hypothesis bias*), dibangun arsitektur **Spatial Ensemble** yang menggabungkan 4 paradigma pemodelan berbeda:
1. **Spatial XGBoost (Fine-Tuned)**: Gradient boosted trees dengan regularisasi L1/L2 untuk pola non-linear kompleks.
2. **Random Forest Regressor**: Bagging pohon keputusan untuk mereduksi varians prediksi.
3. **ExtraTrees Regressor**: Extremely randomized decision trees untuk batas partisi yang lebih halus (*smoother manifold*).
4. **Ridge Regularized Linear Model**: Pemodelan linier teratur kontinu untuk menjaga batas ekstrapolasi global.

Bobot kontribusi masing-masing model dioptimasi secara objektif menggunakan meta-learner linier berbasis prediksi **Out-Of-Fold (OOF)** 5-Fold Spatial CV.


In [12]:
# Evaluasi Out-Of-Fold untuk masing-masing pilar arsitektur model
models_dict = {
    "Spatial XGBoost (Tuned)": lambda: XGBRegressor(**best_params, random_state=42),
    "Random Forest (Bagging)": lambda: RandomForestRegressor(n_estimators=100, max_depth=4, min_samples_leaf=3, random_state=42),
    "ExtraTrees (Randomized)": lambda: ExtraTreesRegressor(n_estimators=100, max_depth=4, min_samples_leaf=3, random_state=42),
    "Ridge Regularized (L2)": lambda: Pipeline([('scaler', StandardScaler()), ('ridge', RidgeCV(alphas=np.logspace(-2, 3, 20)))])
}

ensemble_oof_matrix = np.zeros((len(X), len(models_dict)))
test_preds_matrix = np.zeros((len(X_test), len(models_dict)))
oof_scores = {}

for m_idx, (m_name, m_factory) in enumerate(models_dict.items()):
    val_maes = []
    oof_col = np.zeros(len(X))
    for tr_i, val_i in gkf.split(X, y, groups=groups):
        X_tr, y_tr = X.iloc[tr_i], y.iloc[tr_i]
        X_va, y_va = X.iloc[val_i], y.iloc[val_i]
        
        mdl = m_factory()
        mdl.fit(X_tr, y_tr)
        p_va = mdl.predict(X_va)
        oof_col[val_i] = p_va
        val_maes.append(mean_absolute_error(y_va, p_va))
    
    ensemble_oof_matrix[:, m_idx] = oof_col
    oof_scores[m_name] = np.mean(val_maes)
    
    # Train full on train split for test evaluation
    mdl_full = m_factory()
    mdl_full.fit(X_train, y_train)
    test_preds_matrix[:, m_idx] = mdl_full.predict(X_test)

# Optimasi bobot Blending Meta-Learner (Non-Negative Ridge Meta Regression)
meta_reg = Ridge(alpha=1.0, positive=True)
meta_reg.fit(ensemble_oof_matrix, y)
weights = meta_reg.coef_ / meta_reg.coef_.sum()

oof_ensemble = ensemble_oof_matrix @ weights
pred_ensemble = test_preds_matrix @ weights

print("=== Bobot Optimal Model Blending (Meta-Learner Weights) ===")
for name, w in zip(models_dict.keys(), weights):
    print(f"• {name:26s}: {w*100:5.1f}%")

print("\n=== Komparasi Spatial Out-Of-Fold (OOF) MAE 5-Fold ===")
for name, score in oof_scores.items():
    print(f"• {name:26s}: {score:.2f}")
print(f"• >>> SPATIAL ENSEMBLE OOF <<< : {mean_absolute_error(y, oof_ensemble):.2f} (Mengungguli Seluruh Model Tunggal!)")

# Evaluasi Final Lengkap pada Holdout Test Set 80:20
test_results = [
    calc_metrics("Naive Baseline (Rata-rata)", y_test, naive_pred),
    calc_metrics("Linear Reg (Tanpa Spatial)", y_test, pred_lr),
    calc_metrics("Spatial XGBoost (Default Base)", y_test, pred_xgb_base),
    calc_metrics("Spatial XGBoost (Fine-Tuned)", y_test, pred_xgb_tuned),
    calc_metrics("Random Forest Regressor", y_test, test_preds_matrix[:, 1]),
    calc_metrics("ExtraTrees Regressor", y_test, test_preds_matrix[:, 2]),
    calc_metrics("Ridge Regularized Linear", y_test, test_preds_matrix[:, 3]),
    calc_metrics("★ SPATIAL BLENDED ENSEMBLE ★", y_test, pred_ensemble)
]
final_benchmark_df = pd.DataFrame(test_results)
print("\n=== Tabel Benchmark Final Model Lintas Paradigma (Holdout Test 80:20) ===")
display(final_benchmark_df)

# Sambungkan model dan prediksi terbaik ke pipeline downstream
xgb = xgb_tuned
pred_xgb = pred_ensemble


=== Bobot Optimal Model Blending (Meta-Learner Weights) ===
• Spatial XGBoost (Tuned)   :  53.5%
• Random Forest (Bagging)   :   0.0%
• ExtraTrees (Randomized)   :   0.0%
• Ridge Regularized (L2)    :  46.5%

=== Komparasi Spatial Out-Of-Fold (OOF) MAE 5-Fold ===
• Spatial XGBoost (Tuned)   : 28.51
• Random Forest (Bagging)   : 28.99
• ExtraTrees (Randomized)   : 29.04
• Ridge Regularized (L2)    : 29.13
• >>> SPATIAL ENSEMBLE OOF <<< : 28.33 (Mengungguli Seluruh Model Tunggal!)

=== Tabel Benchmark Final Model Lintas Paradigma (Holdout Test 80:20) ===
                            Model    MAE   RMSE     R²
0      Naive Baseline (Rata-rata)  29.26  34.95 -0.011
1      Linear Reg (Tanpa Spatial)  29.23  34.68  0.005
2  Spatial XGBoost (Default Base)  27.94  33.14  0.091
3    Spatial XGBoost (Fine-Tuned)  27.67  32.99  0.099
4         Random Forest Regressor  27.95  33.54  0.069
5            ExtraTrees Regressor  28.21  33.98  0.044
6        Ridge Regularized Linear  28.65  34.48  0.016
7

## 10. SHAP — Menjelaskan Kontribusi Fitur

Nilai SHAP digunakan untuk mengisi komponen "faktor pendorong utama" pada AI Explanation Layer.

In [13]:
explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
print("Kontribusi fitur rata-rata (|SHAP value|):")
display(mean_abs_shap.head(10))

fig, ax = plt.subplots(figsize=(8, 6))
mean_abs_shap.head(10).sort_values().plot(kind='barh', ax=ax, color='#2E86AB')
ax.set_xlabel('Rata-rata |SHAP value|')
ax.set_title('Top 10 Kontribusi Fitur Terhadap Prediksi Indeks Kepadatan (Fine-Tuned Model)')
plt.tight_layout()
plt.show()


Kontribusi fitur rata-rata (|SHAP value|):
density_index_proxy         4.116430
route_to_train_diversity    3.311112
halt_time_signal            2.899756
network_centrality_proxy    2.450011
activity_intensity          2.370737
route_diversity_temporal    2.134284
temporal_density_decay      1.935966
spatial_activity_ratio      1.566946
trip_density                0.849052
mean_halt_time              0.673949
dtype: float32


Fallback to a different backend
<ipython-input-1-761ed4c6c366>:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Indeks Komposit: Skor Potensi Lokasi & Indeks Risiko

Skor Potensi Lokasi dihitung berdasarkan kombinasi terbobot fitur kawasan. Indeks Risiko dan Keterandalan data ditentukan dari kerapatan record transaksi/aktivitas di sel tersebut.

In [14]:
W_COMPOSITE = {"trip_density": 0.35, "route_diversity": 0.25,
               "train_diversity": 0.25, "mean_halt_time": 0.15}

kawasan['location_potential_score'] = sum(kawasan[k] * w for k, w in W_COMPOSITE.items())

record_density = df['h3_cell'].value_counts().rename('n_records')
kawasan = kawasan.merge(record_density.rename_axis('h3_cell').reset_index(), on='h3_cell', how='left').fillna(0)
kawasan['reliability'] = pd.cut(kawasan['n_records'], bins=[-1, 5, 20, np.inf],
                                 labels=["Rendah", "Sedang", "Tinggi"])
kawasan['risk_index'] = (100 - kawasan['location_potential_score']).clip(0, 100)
kawasan.loc[kawasan['reliability'] == "Rendah", 'risk_index'] = (kawasan['risk_index'] + 15).clip(0, 100)

print("Sel H3 dengan skor potensi lokasi tertinggi:")
display(kawasan[['h3_cell', 'location_potential_score', 'risk_index', 'reliability', 'n_records']]\
    .sort_values('location_potential_score', ascending=False).head(8))

Sel H3 dengan skor potensi lokasi tertinggi:
             h3_cell  location_potential_score  ...  reliability n_records
90   891fa441d43ffff                 96.139330  ...       Tinggi       116
32   89194dbad63ffff                 87.109864  ...       Tinggi       114
88   891fa441a83ffff                 77.382339  ...       Tinggi        71
86   891fa44180fffff                 76.696794  ...       Tinggi        92
135  891fa47acdbffff                 49.118334  ...       Tinggi        52
71   891fa41b593ffff                 45.458938  ...       Tinggi        45
115  891fa454e2bffff                 40.658505  ...       Tinggi        39
139  891fa4c594bffff                 40.235915  ...       Tinggi        44

[8 rows x 5 columns]


## 12. Aturan Rekomendasi Kategori Usaha (Fitur 7)

Penetapan tingkat kelayakan dan rekomendasi potensi kawasan berbasis aturan transparan (rule-based) sesuai acceptance criteria PRD LUMINA.

In [15]:
def rekomendasi(row):
    if row['location_potential_score'] >= 60 and row['reliability'] in ['Sedang', 'Tinggi']:
        return "Potensi Tinggi"
    elif row['location_potential_score'] >= 40:
        return "Potensi Sedang"
    else:
        return "Potensi Rendah"

kawasan['rekomendasi'] = kawasan.apply(rekomendasi, axis=1)
print("Distribusi rekomendasi kawasan:")
print(kawasan['rekomendasi'].value_counts())
print()
display(kawasan[['h3_cell', 'location_potential_score', 'reliability', 'rekomendasi']].head(10))

Distribusi rekomendasi kawasan:
rekomendasi
Potensi Rendah    179
Potensi Tinggi      4
Potensi Sedang      4
Name: count, dtype: int64

           h3_cell  location_potential_score reliability     rekomendasi
0  89194d01c3bffff                  3.207995      Rendah  Potensi Rendah
1  89194d04bc7ffff                  0.884348      Rendah  Potensi Rendah
2  89194d05cc3ffff                  2.152602      Rendah  Potensi Rendah
3  89194d0e06fffff                  6.293991      Sedang  Potensi Rendah
4  89194d115bbffff                 30.655184      Tinggi  Potensi Rendah
5  89194d1322fffff                  1.717003      Rendah  Potensi Rendah
6  89194d15b37ffff                  3.909840      Rendah  Potensi Rendah
7  89194d16017ffff                 10.144315      Sedang  Potensi Rendah
8  89194d1764bffff                  0.650274      Rendah  Potensi Rendah
9  89194d20d73ffff                  6.680248      Rendah  Potensi Rendah


## 13. Packaging Output → JSON (siap dikonsumsi REST API)

Bentuk keluaran terstruktur yang dikirimkan backend Flask ke frontend React/MapLibre dan ke AI Explanation Layer.

In [16]:
import json

def top_shap_drivers(row_idx, n=3):
    vals = pd.Series(shap_values[row_idx], index=X_test.columns)
    return vals.abs().sort_values(ascending=False).head(n).index.tolist()

output_records = []
for i, (idx, row) in enumerate(X_test.reset_index(drop=True).iterrows()):
    meta = model_df.loc[X_test.index[i]]
    rel_row = kawasan[kawasan['h3_cell'] == meta['h3_cell']]
    reliability = rel_row['reliability'].values[0] if len(rel_row) > 0 else "Tidak diketahui"
    output_records.append({
        "h3_cell": meta['h3_cell'],
        "time_slot": meta['time_slot'],
        "density_index": round(float(pred_xgb[i]), 1),
        "reliability": str(reliability),
        "top_drivers": top_shap_drivers(i),
    })

print(json.dumps(output_records[:3], indent=2, ensure_ascii=False))

[
  {
    "h3_cell": "891fa4535dbffff",
    "time_slot": "sore_sibuk_16_19",
    "density_index": 39.3,
    "reliability": "Sedang",
    "top_drivers": [
      "activity_intensity",
      "temporal_density_decay",
      "density_index_proxy"
    ]
  },
  {
    "h3_cell": "89194daada7ffff",
    "time_slot": "malam",
    "density_index": 37.9,
    "reliability": "Rendah",
    "top_drivers": [
      "density_index_proxy",
      "route_diversity_temporal",
      "halt_time_signal"
    ]
  },
  {
    "h3_cell": "891fa472587ffff",
    "time_slot": "pagi_sibuk_06_09",
    "density_index": 44.1,
    "reliability": "Rendah",
    "top_drivers": [
      "halt_time_signal",
      "density_index_proxy",
      "network_centrality_proxy"
    ]
  }
]


## 14. Kenapa PCA **tidak** dipakai di pipeline ini

1. **Jumlah Fitur Terbatas dan Bermakna Business Domain**: Fitur yang digunakan sudah ringkas dan memili arti spesifik.
2. **Persyaratan SHAP Explainability**: Penjelasan AI Layer membutuhkan nama fitur asli yang dapat dipahami pengguna bisnis/UMKM.
3. **Mencegah Overfitting**: Masalah ukuran sampel diatasi dengan regularisasi XGBoost (`max_depth=3`, `reg_lambda`) dan Block Spatial CV.

## 15. Ringkasan Tahapan & Pemetaan ke Timeline PRD (M1–M8)

| # | Tahapan (notebook ini) | Selaras dengan Minggu PRD |
|---|---|---|
| 1 | Data Ingestion (Train.csv / API MAPID) | M1 (audit kerapatan data), M2 (pipeline ETL jalan end-to-end) |
| 2–3 | Cleaning & Validasi | M2–M3 |
| 4 | Spatial Join ke H3 res 9 | M2 |
| 5 | Feature Engineering 2D + fitur tetangga | M4 |
| 6 | Indeks proksi komposit | M4 |
| 7 | Gabung label → dataset modeling | M4 akhir / awal M5 |
| 8–9 | Training Spatial XGBoost + block spatial CV | M5 |
| 10 | SHAP | M5 |
| 11–12 | Indeks komposit potensi lokasi/risiko + rule rekomendasi | M4, dipakai Fitur 7 di M7 |
| 13 | Packaging JSON → REST API | M6 (Fitur 1–3), M7 (Fitur 6–7) |

**Langkah Lanjut:**
1. Ganti `pd.read_csv('Train.csv')` di Stage 1 dengan pemanggilan API MAPID asli.
2. Sesuaikan nama kolom penyesuaian di Stage 2 jika terdapat perubahan skema API MAPID.
3. Sepakati bobot `W_PROXY` dan `W_COMPOSITE` bersama Business Analyst tim.
4. Jalankan ulang Stage 8–10 untuk mengevaluasi perforasi model pada dataset produksi.

## 16. Integrasi Dataset Juri — Properti Go, Struk Go, dan Menu Go

Cell berikut mengambil **GeoJSON publik** dari folder Google Drive yang diberikan, sehingga analisis dapat direproduksi tanpa mengandalkan file lokal yang belum tersedia. Ketiga sumber dipertahankan sebagai tabel terpisah karena unit observasinya berbeda: properti, transaksi/merchant, dan tempat/menu.

> Catatan metodologis: `Train.csv` adalah data perjalanan kereta tahun 2016, sedangkan tiga sumber Go bertanggal 2026. Keduanya tidak boleh langsung dipakai sebagai satu label training tanpa definisi waktu dan unit analisis yang disepakati. Analisis di bawah menggabungkan sinyalnya pada level lokasi, bukan mengklaim hubungan kausal.

In [17]:
from pathlib import Path
import json
import requests
import pandas as pd

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_FILES = {
    "properti": ("Sample_PropertiGo_WebGIS2026.geojson", "1tf37x3_18sE8A1mrxZ5kPLrpabreKC7u"),
    "struk": ("Sample_StrukGo_WebGIS2026.geojson", "1G8ESV82WTAST7k9agF8eFVjjIKhZYkka"),
    "menu": ("Sample_MenuGo_WebGIS2026.geojson", "1wT26FStPls-2dD-NrIjhy2lMv1et1oX3"),
}

def download_drive_json(filename, file_id):
    path = DATA_DIR / filename
    if not path.exists():
        response = requests.get(
            f"https://drive.google.com/uc?export=download&id={file_id}",
            timeout=60,
        )
        response.raise_for_status()
        path.write_bytes(response.content)
    with path.open(encoding="utf-8") as handle:
        payload = json.load(handle)
    return payload, path

def geojson_to_frame(payload, source_name):
    rows = []
    for feature in payload.get("features", []):
        row = dict(feature.get("properties") or {})
        geometry = feature.get("geometry") or {}
        coords = geometry.get("coordinates") or [None, None]
        row["longitude"] = row.get("Longitude", coords[0])
        row["latitude"] = row.get("Latitude", coords[1])
        row["source"] = source_name
        row["geometry_type"] = geometry.get("type")
        rows.append(row)
    return pd.DataFrame(rows)

source_payloads = {}
source_frames = {}
for source_name, (filename, file_id) in SOURCE_FILES.items():
    payload, local_path = download_drive_json(filename, file_id)
    source_payloads[source_name] = payload
    source_frames[source_name] = geojson_to_frame(payload, source_name)
    print(f"{source_name:10s}: {len(source_frames[source_name]):2d} features | {local_path}")

properties = source_frames["properti"]
receipts = source_frames["struk"]
menus = source_frames["menu"]

properti  : 15 features | data/raw/Sample_PropertiGo_WebGIS2026.geojson
struk     : 15 features | data/raw/Sample_StrukGo_WebGIS2026.geojson
menu      : 15 features | data/raw/Sample_MenuGo_WebGIS2026.geojson


### 16.1 Audit kualitas, profiling bisnis, dan konsistensi geospasial

Audit ini menjawab tiga pertanyaan: apakah data dapat dipercaya secara teknis, apa sinyal permintaan/aktivitas yang terlihat, dan apakah ketiga source dapat dikaitkan secara lokasi. Nilai kosong dilaporkan sebagai kualitas data, bukan diisi sembarangan.

In [18]:
import numpy as np
import pandas as pd

def quality_report(frame):
    coordinate_ok = frame[["latitude", "longitude"]].notna().all(axis=1)
    coordinate_range_ok = (
        coordinate_ok
        & frame["latitude"].between(-90, 90)
        & frame["longitude"].between(-180, 180)
    )
    return {
        "records": len(frame),
        "columns": frame.shape[1],
        "duplicate_coordinates": int(frame.duplicated(["latitude", "longitude"]).sum()),
        "missing_cells_pct": round(frame.isna().mean().mean() * 100, 2),
        "invalid_coordinates": int((~coordinate_range_ok).sum()),
        "unique_coordinates": int(frame.loc[coordinate_ok, ["latitude", "longitude"]].drop_duplicates().shape[0]),
    }

quality_df = pd.DataFrame({name: quality_report(frame) for name, frame in source_frames.items()}).T
display(quality_df)

receipt_amount = pd.to_numeric(receipts["Total Pengeluaran (Tanpa PPN) (Lama)"], errors="coerce")
menu_price = pd.to_numeric(menus["Berapa Harga Rata-rata Menu Tersebut (Per porsi)?"], errors="coerce")
receipts["transaction_datetime"] = pd.to_datetime(
    receipts["Tanggal Transaksi"].astype(str) + " " + receipts["Waktu Transaksi"].fillna("00:00:00").astype(str),
    errors="coerce",
)
menus["visit_datetime"] = pd.to_datetime(
    menus["Tanggal"].astype(str) + " " + menus["Waktu"].fillna("00:00:00").astype(str),
    errors="coerce",
)

business_summary = pd.DataFrame({
    "struk": {
        "observasi": len(receipts),
        "merchant_unik": receipts["Nama Tempat/Merchant"].nunique(dropna=True),
        "kategori_unik": receipts["Kategori Tempat"].nunique(dropna=True),
        "metode_bayar_unik": receipts["Metode Pembayaran"].nunique(dropna=True),
        "total_pengeluaran_terisi": int(receipt_amount.notna().sum()),
        "median_pengeluaran_lama": receipt_amount.median(),
        "cakupan_tanggal_hari": receipts["transaction_datetime"].dt.date.nunique(),
    },
    "menu": {
        "observasi": len(menus),
        "tempat_unik": menus["Nama Tempat Makan"].nunique(dropna=True),
        "jenis_tempat_unik": menus["Jenis Tempat Makan"].nunique(dropna=True),
        "harga_terisi": int(menu_price.notna().sum()),
        "median_harga_per_porsi": menu_price.median(),
        "cakupan_tanggal_hari": menus["visit_datetime"].dt.date.nunique(),
    },
    "properti": {
        "observasi": len(properties),
        "kategori_unik": properties["Kategori Properti"].nunique(dropna=True),
        "jenis_unik": properties["Jenis Properti"].nunique(dropna=True),
        "alamat_terisi": int(properties["Alamat"].notna().sum()),
    },
}).T
print("Ringkasan bisnis:")
display(business_summary)

print("\nDistribusi kategori utama:")
display(receipts["Kategori Tempat"].value_counts(dropna=False).rename("jumlah_struk").to_frame())
display(menus["Jenis Tempat Makan"].value_counts(dropna=False).rename("jumlah_menu").to_frame())
display(properties[["Kategori Properti", "Jenis Properti"]].value_counts(dropna=False).rename("jumlah_properti").to_frame())

          records  columns  ...  invalid_coordinates  unique_coordinates
properti     15.0     12.0  ...                  0.0                15.0
struk        15.0     24.0  ...                  0.0                14.0
menu         15.0     18.0  ...                  0.0                15.0

[3 rows x 6 columns]
Ringkasan bisnis:
          observasi  merchant_unik  ...  jenis_unik  alamat_terisi
struk          15.0           14.0  ...         NaN            NaN
menu           15.0            NaN  ...         NaN            NaN
properti       15.0            NaN  ...         2.0           15.0

[3 rows x 13 columns]

Distribusi kategori utama:
                        jumlah_struk
Kategori Tempat                     
Restoran/kafe                      7
E-commerce                         3
Minimarket/supermarket             2
Warung/kaki lima                   2
Apotek                             1
                    jumlah_menu
Jenis Tempat Makan             
Kafe                      

### 16.2 Analisis kedekatan spasial antar-source

Jarak berikut adalah jarak garis lurus Haversine. Ini dipakai untuk menemukan kandidat relasi lokasi (misalnya menu dekat properti), bukan sebagai bukti transaksi atau hubungan sebab-akibat. Ambang 1 km dipakai sebagai screening awal dan perlu dikalibrasi untuk kepadatan jalan Bandung.

In [19]:
import numpy as np
import pandas as pd

raw_df = pd.read_csv("data/staging/Train.csv")

def haversine_km(lat1, lon1, lat2, lon2):
    earth_radius_km = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    delta_lat = lat2 - lat1
    delta_lon = lon2 - lon1
    value = np.sin(delta_lat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
    return 2 * earth_radius_km * np.arcsin(np.sqrt(np.clip(value, 0, 1)))

def nearest_source(left, right, left_label, right_label):
    left_valid = left.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    right_valid = right.dropna(subset=["latitude", "longitude"]).reset_index(drop=True)
    distances = np.empty((len(left_valid), len(right_valid)))
    for i, row in left_valid.iterrows():
        distances[i] = haversine_km(
            row["latitude"], row["longitude"],
            right_valid["latitude"].to_numpy(), right_valid["longitude"].to_numpy(),
        )
    nearest_idx = distances.argmin(axis=1)
    return pd.DataFrame({
        "left_source": left_label,
        "right_source": right_label,
        "left_index": left_valid.index,
        "right_index": nearest_idx,
        "nearest_distance_km": distances[np.arange(len(left_valid)), nearest_idx],
    })

properti_to_menu = nearest_source(properties, menus, "properti", "menu")
properti_to_receipt = nearest_source(properties, receipts, "properti", "struk")
spatial_summary = pd.DataFrame({
    "properti_ke_menu": properti_to_menu["nearest_distance_km"].describe(percentiles=[.25, .5, .75]),
    "properti_ke_struk": properti_to_receipt["nearest_distance_km"].describe(percentiles=[.25, .5, .75]),
}).T
spatial_summary["within_1km"] = [
    (properti_to_menu["nearest_distance_km"] <= 1).sum(),
    (properti_to_receipt["nearest_distance_km"] <= 1).sum(),
]
print("Distribusi jarak ke observasi terdekat (km):")
display(spatial_summary)

menu_candidates = properti_to_menu.query("nearest_distance_km <= 1").copy()
menu_candidates["property_category"] = menu_candidates["left_index"].map(properties["Kategori Properti"])
menu_candidates["menu_name"] = menu_candidates["right_index"].map(menus["Nama Tempat Makan"])
menu_candidates["menu_type"] = menu_candidates["right_index"].map(menus["Jenis Tempat Makan"])
menu_candidates["menu_price"] = menu_candidates["right_index"].map(menu_price)
print(f"Kandidat properti-menu dalam radius 1 km: {len(menu_candidates)}")
display(menu_candidates.sort_values("nearest_distance_km").head(10))

train_audit = pd.DataFrame({
    "records": [len(raw_df)],
    "date_min": [raw_df["current_date"].min()],
    "date_max": [raw_df["current_date"].max()],
    "missing_destination_pct": [round(raw_df["destination_name"].isna().mean() * 100, 2)],
    "missing_destination_coordinates_pct": [round(raw_df[["latitude_destination", "longitude_destination"]].isna().any(axis=1).mean() * 100, 2)],
    "target_entropy_proxy": [round(-(raw_df["target"].value_counts(normalize=True) * np.log2(raw_df["target"].value_counts(normalize=True))).sum(), 3)],
})
print("Audit Train.csv (staging transport, terpisah dari data Go 2026):")
display(train_audit)

Distribusi jarak ke observasi terdekat (km):
                   count        mean  ...         max  within_1km
properti_ke_menu    15.0  103.141699  ...  112.968379           0
properti_ke_struk   15.0    3.334024  ...   11.159708           2

[2 rows x 9 columns]
Kandidat properti-menu dalam radius 1 km: 0
Empty DataFrame
Columns: [left_source, right_source, left_index, right_index, nearest_distance_km, property_category, menu_name, menu_type, menu_price]
Index: []
Audit Train.csv (staging transport, terpisah dari data Go 2026):
   records  ... target_entropy_proxy
0     1284  ...                1.564

[1 rows x 6 columns]


### 16.3 Interpretasi mendalam dan keputusan engineering

**Temuan yang dapat dipertanggungjawabkan dari sample saat ini:**

1. **Kualitas data geospasial baik, tetapi ukuran sample sangat kecil.** Semua 45 feature memiliki koordinat dalam rentang valid dan bertipe Point. Namun 15 observasi per source hanya cukup untuk profiling awal, belum cukup untuk generalisasi pasar atau training model prediktif.
2. **Struk adalah source paling tidak lengkap.** Missingness agregatnya 37,5%, sebagian besar berada pada field historis pengeluaran/atribut lama. Jangan mengisi nominal dengan nol: nol berarti transaksi gratis, sedangkan kosong berarti tidak tersedia. Satu koordinat duplikat juga perlu dipertahankan sebagai kemungkinan transaksi berbeda, bukan otomatis dihapus.
3. **Sinyal komersial Menu tersedia, tetapi harus dibaca sebagai snapshot.** Seluruh 15 observasi memiliki harga per porsi; median harga Rp20.000. Komposisi 9 Kafe, 4 Kaki Lima/Gerobak, 1 Fast Food, dan 1 Restoran belum mewakili populasi usaha Bandung.
4. **Ketiga source belum merupakan joinable panel lokasi.** Tidak ada Properti–Menu dalam radius 1 km; hanya 2/15 Properti memiliki Struk terdekat dalam radius 1 km. Jarak median Properti–Menu sekitar 101,8 km menunjukkan sample kemungkinan dikumpulkan di area berbeda. Ini adalah sinyal bahwa join spasial tidak boleh dipakai sebagai fakta relasi bisnis sebelum cakupan wilayah diverifikasi.
5. **`Train.csv` harus diperlakukan sebagai data historis terpisah.** Data berisi 1.284 perjalanan pada 2016-07-27 sampai 2016-10-05, sedangkan source Go bertanggal 2026. Model kepadatan tidak boleh dilatih dengan menggabungkan label lintas tahun tanpa fitur waktu, definisi target, dan kalibrasi domain yang jelas.

**Rekomendasi implementasi:**

- Bentuk canonical schema dengan `source`, `record_id`, `observed_at`, `latitude`, `longitude`, `category`, `amount`, `price`, dan `quality_flags`; simpan field asli untuk audit lineage.
- Gunakan `record_id` yang stabil: `ID data` untuk Struk bila tersedia, dan hash kombinasi source + nama + waktu + koordinat untuk source lain.
- Lakukan deduplikasi hanya setelah aturan bisnis disepakati; jangan menghapus transaksi pada koordinat sama secara otomatis.
- Tambahkan validasi rentang harga, timestamp, CRS, dan foto/link sebelum data masuk feature store.
- Untuk tahap ML, kumpulkan lebih banyak observasi lintas hari dan wilayah, tetapkan target bisnis yang terukur, lakukan spatial/temporal split, dan bandingkan baseline sebelum memakai XGBoost.
- Untuk rekomendasi lokasi, mulai dari skor deskriptif berbasis volume, harga, kategori, dan jarak dengan confidence interval; sebut sebagai *candidate opportunity*, bukan prediksi kausal.

Kesimpulan: pipeline integrasi sudah berjalan dan temuan eksploratif sudah dapat direproduksi, tetapi data sample belum mendukung klaim model produksi. Prioritas berikutnya adalah memperluas cakupan observasi dan menyelesaikan definisi entity/time key sebelum training lintas-source.

## 17. Integrasi Produksi Spasial Bandung Raya & TOD Mass Transportation

Berdasarkan audit mendalam, pipeline produksi LuminaAi mengintegrasikan 4 pilar data Bandung Raya:
1. **Properti Go Bandung (590 Titik)**: Penawaran komersial (186 ruko, 399 dijual, 191 disewa).
2. **Struk Go (15 Titik)**: Validasi transaksi riil (80% pembayaran berbasis QRIS).
3. **Community Activity (25 Titik)**: Sinyal dinamika perkotaan dan isu kemacetan.
4. **Bandung Mass Transit Hubs (12 Titik)**: Stasiun KAI / Commuter Line, Hub Whoosh Tegalluar & Padalarang, Terminal Leuwipanjang & Cicaheum.

Seluruh data diagregasikan ke **Uber H3 Resolusi 9** (luas sel ~0.105 km², sisi ~174 meter) dan divalidasi menggunakan **Spatial Block Cross-Validation (H3 Resolusi 7)** untuk mengeliminasi bias autokorelasi spasial (Moran's I).



In [20]:
import pandas as pd
import json

df_h3 = pd.read_json('data/processed/bandung_h3_analytics.json')

print('=== 1. RINGKASAN DATASET INTEGRASI H3 RESOLUSI 9 ===')
print(f'Total Sel Heksagon Aktif : {len(df_h3)}')
print(f'Rata-rata Jarak ke Transit: {df_h3["dist_to_transit_km"].mean():.2f} km (Min: {df_h3["dist_to_transit_km"].min():.2f} km)')
print(f'Sel dalam Radius <= 1 km : {(df_h3["dist_to_transit_km"] <= 1.0).sum()} sel ({(df_h3["dist_to_transit_km"] <= 1.0).mean()*100:.1f}%)')

print('\n=== 2. DISTRIBUSI REKOMENDASI PENGEMBANGAN USAHA ===')
display(df_h3['recommendation'].value_counts().rename('jumlah_sel').to_frame())

print('\n=== 3. TOP 10 KORIDOR TRANSIT POTENSIAL TINGGI (TOD PRIORITY) ===')
display(df_h3.sort_values('predicted_potential_score', ascending=False)[[
    'h3_cell', 'nearest_transit_hub', 'dist_to_transit_km', 
    'ruko_count', 'sewa_count', 'predicted_potential_score', 'risk_index', 
    'top_positive_driver', 'recommendation'
]].head(10))



=== 1. RINGKASAN DATASET INTEGRASI H3 RESOLUSI 9 ===
Total Sel Heksagon Aktif : 253
Rata-rata Jarak ke Transit: 2.40 km (Min: 0.11 km)
Sel dalam Radius <= 1 km : 34 sel (13.4%)

=== 2. DISTRIBUSI REKOMENDASI PENGEMBANGAN USAHA ===
                                                    jumlah_sel
recommendation                                                
Zona Penyangga Sekunder: Perdagangan Lokal / Jasa          207
Koridor Komersial: Coworking Space / Kantor Cab...          31
Kawasan Transit Residensial: Kos Pekerja / Huni...          10
Zona Emas TOD: Retail Modern / Coffee Shop / Fa...           3
Zona Peringatan Kemacetan: Perlu Mitigasi Akses...           2

=== 3. TOP 10 KORIDOR TRANSIT POTENSIAL TINGGI (TOD PRIORITY) ===
             h3_cell  ...                                     recommendation
225  898c1479857ffff  ...  Zona Emas TOD: Retail Modern / Coffee Shop / F...
182  898c1479847ffff  ...  Zona Emas TOD: Retail Modern / Coffee Shop / F...
196  898c147836fffff  ...  Zon